# "hey TARS" in real people's voices

Clones Mozilla Common Voice speakers (CC0, through the Mozilla Data Collective) with Chatterbox Turbo (MIT; its Perth
watermark is kept) saying six "hey TARS" variants and six lookalikes. TARS uses the clips to evaluate wake models on
many real speakers, and can train on a separate set of speakers. See `docs/wake-word.md` and `training/README.md`.

The clips stay private: the datasets' terms forbid rehosting the audio and trying to identify speakers.

Needs: a GPU runtime (a T4 is enough: about a second a clip), `MDC_API_KEY` in Colab Secrets, and each dataset's terms
accepted on its page at mozilladatacollective.com. The Data Collective has a daily download quota.

In [ ]:
!pip -q install chatterbox-tts==0.1.7 "setuptools<81" soundfile
!pip -q uninstall -y torchvision  # built for Colab's own torch; chatterbox pins torch 2.6
import csv
import hashlib
import json
import random
import subprocess
import sys
import tarfile
import time
from pathlib import Path

from google.colab import drive, runtime, userdata

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/tars-cloned")
WORK = Path("/content/work")
for p in (DRIVE, WORK / "jobs"):
    p.mkdir(parents=True, exist_ok=True)
KEY = userdata.get("MDC_API_KEY")
!nvidia-smi --query-gpu=name,memory.total --format=csv

## The cloner

One process clones one batch of 50 voices and exits, so the memory Chatterbox accumulates is released between batches.
Each batch is written to Drive as one archive; batches already there are skipped, so a rerun resumes.

In [ ]:
%%writefile /content/clone.py
"""Clones the next unfinished batch of 50 voices in a speaker list into DRIVE/<list>-<batch>.tar, then exits.
Batch numbers given after the two paths are skipped."""
import io, json, sys, tarfile, time
from pathlib import Path
import librosa, numpy as np, soundfile as sf, torch

_resample = librosa.resample
librosa.resample = lambda *a, **k: _resample(*a, **k).astype(np.float32)  # Chatterbox expects float32

POSITIVE = ["Hey Tarss.", "Hey Tarss!", "Hey TARS.", "Hey, TARS?", "Hey Tars!", "hey tars"]
LOOKALIKE = ["Hey stars.", "Hey cars.", "Hey Tara.", "Hey bars.", "Hey guards.", "Hey tarot."]
BATCH = 50


def reference(clips: list[str]) -> np.ndarray | None:
    """5-12 s of the speaker's own speech, silences trimmed; None if they recorded too little."""
    parts, total = [], 0.0
    for c in clips:
        try:
            part = librosa.effects.trim(librosa.load(c, sr=24000)[0], top_db=30)[0]
        except Exception:
            continue
        parts += [part, np.zeros(4800, np.float32)]
        total += len(part) / 24000 + 0.2
        if total >= 8.0:
            break
    return np.concatenate(parts)[: 24000 * 12] if total >= 5.5 else None


def main(job: str, out_dir: str, skip: set[int]) -> int:
    voices, out = json.loads(Path(job).read_text()), Path(out_dir)
    done = {p.stem for p in out.glob("*.tar")}
    todo = [b for b in range(0, len(voices), BATCH) if b // BATCH not in skip and f"{Path(job).stem}-{b // BATCH:04d}" not in done]
    if not todo:
        return 0
    from chatterbox.models.s3tokenizer.s3tokenizer import S3Tokenizer
    from chatterbox.tts_turbo import ChatterboxTurboTTS

    mel = S3Tokenizer.log_mel_spectrogram
    S3Tokenizer.log_mel_spectrogram = lambda self, audio, padding=0: mel(self, torch.as_tensor(audio).float(), padding)
    model = ChatterboxTurboTTS.from_pretrained(device="cuda")
    b, t0, n = todo[0], time.time(), 0
    name = f"{Path(job).stem}-{b // BATCH:04d}"
    buf = io.BytesIO()
    with tarfile.open(fileobj=buf, mode="w") as tar:
        for v in voices[b : b + BATCH]:
            ref = reference(v["clips"])
            if ref is None:
                continue
            ref_path = f"/tmp/ref_{Path(job).stem}.wav"
            sf.write(ref_path, ref, 24000)
            model.prepare_conditionals(ref_path)
            for kind, texts in (("positive", POSITIVE), ("near_miss", LOOKALIKE)):
                for k, text in enumerate(texts):
                    wav = model.generate(text).squeeze(0).numpy()
                    clip = io.BytesIO()
                    sf.write(clip, librosa.resample(wav, orig_sr=model.sr, target_sr=16000), 16000, format="WAV",
                             subtype="PCM_16")
                    info = tarfile.TarInfo(f"{kind}/{v['id']}_{k}.wav")
                    info.size = clip.tell()
                    clip.seek(0)
                    tar.addfile(info, clip)
                    n += 1
    (out / f"{name}.tar.part").write_bytes(buf.getvalue())
    (out / f"{name}.tar.part").rename(out / f"{name}.tar")
    print(f"{name}: {n} clips, {(time.time() - t0) / max(n, 1):.2f} s a clip", flush=True)
    return len(todo) - 1


if __name__ == "__main__":
    sys.exit(1 if main(sys.argv[1], sys.argv[2], set(map(int, sys.argv[3:]))) else 0)  # 1: more batches left

## 1. Speakers

Each Common Voice subset is downloaded, its speaker tables and the chosen speakers' clips are unpacked, and the
archive is deleted, so the disk never holds more than one. Up to `PER_SET` speakers with at least 3 clips per subset,
as many women as the subset has up to half, chosen with a fixed seed.

In [ ]:
SETS = {
    "irish": "cmrt71fck001bmm07cj9h312i",
    "scottish": "cmrt717ci0017mm075ewqaw6v",
    "malaysian": "cmrt71n6q001fmm07u64adwuv",
    "us-south": "cmryvrc6i014vo90786kriix5",
    "us-female": "cmrt70j4z001qmm07nvfsmgmr",
    "us-male": "cmrt6zbgx000vmm07hfuefigk",
    "british": "cmrt6zrob000zmm07yqwjlpwi",
    "south-asian": "cmrt70sar001umm07jwxzhw89",
    "australian": "cmrt710620013mm071t45y6wb",
}
USE = ["irish", "scottish", "malaysian", "us-south"]  # the first evaluation set: 486 speakers
PER_SET = 600
import requests


def fetch(name: str, dataset_id: str) -> list[dict]:
    job = WORK / "jobs" / f"{name}.json"
    if job.exists():
        return json.loads(job.read_text())
    meta = requests.post(
        f"https://mozilladatacollective.com/api/datasets/{dataset_id}/download",
        headers={"Authorization": f"Bearer {KEY}", "Content-Type": "application/json"},
    ).json()
    if "downloadUrl" not in meta:
        raise SystemExit(f"{name}: {meta} (are its terms accepted, and is today's quota left?)")
    archive = WORK / meta["filename"]
    subprocess.run(["curl", "-s", "-o", str(archive), meta["downloadUrl"]], check=True)
    folder = WORK / name
    folder.mkdir(exist_ok=True)
    subprocess.run(["tar", "-xzf", str(archive), "-C", str(folder), "--wildcards", "*.tsv"], check=True)
    validated = next(folder.rglob("validated.tsv"), None)
    if validated is None:
        raise SystemExit(f"{name}: no validated.tsv in {meta['filename']}")
    rows = list(csv.DictReader(validated.read_text().splitlines(), delimiter="\t"))
    by = {}
    for r in rows:
        by.setdefault(r["client_id"], []).append(r)
    ok = [v for v in by.values() if len(v) >= 3]
    rng = random.Random(0)
    women = [v for v in ok if v[0]["gender"].startswith("female")]
    rest = [v for v in ok if not v[0]["gender"].startswith("female")]
    rng.shuffle(women), rng.shuffle(rest)
    take = women[: PER_SET // 2]
    take += rest[: PER_SET - len(take)]
    prefix = validated.parent.relative_to(folder)
    (WORK / "members.txt").write_text("\n".join(str(prefix / "clips" / r["path"]) for v in take for r in v[:6]))
    subprocess.run(["tar", "-xzf", str(archive), "-C", str(folder), "-T", str(WORK / "members.txt")], check=True)
    archive.unlink()
    voices = [
        {
            "id": f"{name}-{i:04d}",
            "gender": v[0]["gender"] or "unknown",
            "clips": [str(folder / prefix / "clips" / r["path"]) for r in v[:6]],
        }
        for i, v in enumerate(take)
    ]
    job.write_text(json.dumps(voices))
    print(f"{name}: {len(by)} speakers, {len(ok)} with 3+ clips, {len(take)} chosen")
    return voices


all_voices = [v for name in USE for v in fetch(name, SETS[name])]
print(len(all_voices), "speakers")

## 2. Clone

`WORKERS` processes share the GPU (two fit a T4's memory), each on its share of the speakers, one batch per process.
A batch that fails three times is skipped and its worker goes on with the next one; rerun this cell to try it again.
Rerun this cell after a disconnect too: finished batches are skipped.

In [ ]:
WORKERS, TRIES, BATCH = 2, 3, 50
tag = hashlib.sha1(",".join(v["id"] for v in all_voices).encode()).hexdigest()[:8]
jobs = []
for i in range(WORKERS):
    job = WORK / "jobs" / f"{tag}-worker{i}.json"
    job.write_text(json.dumps(all_voices[i::WORKERS]))
    jobs.append(job)


def next_batch(job: Path, skipped: list[int]) -> int | None:
    for b in range(-(-len(json.loads(job.read_text())) // BATCH)):
        if b not in skipped and not (DRIVE / f"{job.stem}-{b:04d}.tar").exists():
            return b


failures = {job: 0 for job in jobs}
skipped = {job: [] for job in jobs}
current = {}
running = {}
while jobs or running:
    for job in [j for j in jobs if j not in running]:
        current[job] = next_batch(job, skipped[job])
        with open(WORK / f"{job.stem}.log", "a") as log:
            cmd = [sys.executable, "/content/clone.py", str(job), str(DRIVE), *map(str, skipped[job])]
            running[job] = subprocess.Popen(cmd, stdout=log, stderr=log)
    time.sleep(30)
    for job, proc in list(running.items()):
        if proc.poll() is None:
            continue
        del running[job]
        if proc.returncode == 0:
            jobs.remove(job)
        elif proc.returncode == 1:
            failures[job] = 0
        else:
            failures[job] += 1
            if failures[job] >= TRIES:
                skipped[job].append(current[job])
                failures[job] = 0
    print(time.strftime("%H:%M"), len(list(DRIVE.glob(f"{tag}-*.tar"))), "batches on Drive", flush=True)
print("done; skipped batches:", {j.stem: b for j, b in skipped.items() if b})

## 3. Bundle and stop

One archive of every batch on Drive, to download with Drive's own download (or `rclone`), then the runtime is released
so it stops using compute units.

In [ ]:
bundle = DRIVE / "tars-cloned.tar"
with tarfile.open(bundle, "w") as tar:
    for part in sorted(DRIVE.glob(f"{tag}-worker*-*.tar")):
        tar.add(part, arcname=part.name)
print(bundle, f"{bundle.stat().st_size / 1e6:.0f} MB")
runtime.unassign()